# RLHF Clinical Red-Teaming — Pipeline Demo
**Audrey Tjokro · Stephen Dong · Niki Karanikola**

End-to-end demo of all three methods (`baseline` / `dpo` / `ppo`) followed by a test-split evaluation. Every run uses **drastically reduced** hyperparameters — just enough to exercise the full pipeline (data → models → train → eval → artifact sync) so you can sanity-check the wiring before launching paper-grade runs.

This is a **driver only**: no training logic lives here. Each section shells out to `python -m src <method> ...` with an `OVERRIDES` block that shrinks the run to a few minutes.

**Total expected wall-time on a Colab A100:** ~15–25 min for all four sections.

## 1. Mount Drive (optional — used as a local cache for HF + checkpoints)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone repo at a specific commit/branch
Pin a SHA for paper-grade reproducibility. Pass `BRANCH = 'main'` for HEAD.

In [2]:
REPO_URL = 'https://github.com/stephendongg/rlhf-clinical-redteaming.git'
BRANCH   = 'main'
REPO_DIR = '/content/rlhf-clinical-redteaming'

import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
print(subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip())
%cd $REPO_DIR

3458a65e9c7116c77a5ec2ea8fdc0b22f3655bdb
/content/rlhf-clinical-redteaming


## 3. Install requirements

In [3]:
%pip install -q -r requirements.txt
%pip install -q -e .

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 

## 4. Secrets + GCS auth
Add `OPENAI_API_KEY` to Colab Secrets (left sidebar → key icon). The demo overrides each run to use the OpenAI judge (`gpt-4o-mini`) for speed.

In [4]:
from google.colab import userdata, auth
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY').strip()

# GCS auth — uses your Google account; bucket gs://results_043026 must be in project rlhf-clinical-redteaming.
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = 'rlhf-clinical-redteaming'

## 5. Shared demo settings + run helper

`TINY_DATA` shrinks the seed splits to a handful of scenarios. `TINY_GEN` shortens conversations and generations. Each method below adds its own training-loop shrink overrides on top.

If you want to scale up, edit just these dicts (or the per-method `OVERRIDES`).

In [5]:
GCS_BUCKET = 'gs://results_043026'

# Tiny seed splits — full configs default to 100/50/100.
TINY_DATA = [
    'data.n_train=4',
    'data.n_dev=4',
    'data.n_test=4',
]

# Shorter conversations + generations — full configs default to 5 turns / 256 tokens.
TINY_GEN = [
    'max_turns=2',
    'attacker_max_new_tokens=128',
    'target_max_new_tokens=128',
]

DEMO_JUDGE = [
    'judge_backend=openai',
    'judge_model=gpt-4o-mini',
]

EXTRA_FLAGS = ['--allow-dirty']

In [6]:
import shlex, subprocess, time

def run_method(method: str, run_name: str, overrides: list, use_test: bool = False) -> int:
    """Build and stream a `python -m src <method> ...` invocation."""
    cmd = ['python', '-m', 'src', method,
           '--config', f'configs/{method}.yaml',
           '--run-name', run_name,
           '--gcs-bucket', GCS_BUCKET]
    for o in overrides:
        cmd += ['--override', o]
    if use_test:
        cmd.append('--use-test')
    cmd += EXTRA_FLAGS

    print('$', ' '.join(shlex.quote(c) for c in cmd))
    print('=' * 80)
    start = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    elapsed = (time.time() - start) / 60
    print('=' * 80)
    print(f'Finished {method!r} run_name={run_name!r} rc={rc} in {elapsed:.1f} min')
    if rc != 0:
        raise RuntimeError(f'{method} run failed with return code {rc}')
    return rc

## 6. Baseline (dev split)

Untuned attacker vs. target on the dev split. No training; this is the floor that DPO / PPO are trying to beat. Reports ASR, TTF, and effectiveness for n=4 scenarios.

In [8]:
run_method(
    method='baseline',
    run_name='demo-baseline-dev',
    overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE,
)

$ python -m src baseline --config configs/baseline.yaml --run-name demo-baseline-dev --gcs-bucket gs://results_043026 --override data.n_train=4 --override data.n_dev=4 --override data.n_test=4 --override max_turns=2 --override attacker_max_new_tokens=128 --override target_max_new_tokens=128 --override judge_backend=openai --override judge_model=gpt-4o-mini --allow-dirty
08:22:10 INFO    redteam_rlhf.cli | Git SHA: 3458a65e9c7116c77a5ec2ea8fdc0b22f3655bdb
08:22:10 INFO    redteam_rlhf.cli | GPU: NVIDIA A100-SXM4-80GB
08:22:10 INFO    redteam_rlhf.cli | Seeded RNGs with seed=42
08:22:10 INFO    redteam_rlhf.cli | Run ID: f2d9baeb4866 | name=demo-baseline-dev
08:22:10 INFO    redteam_rlhf.results | Run dir: results/runs/f2d9baeb4866
08:22:11 INFO    numexpr.utils | NumExpr defaulting to 12 threads.
08:22:11 INFO    datasets | TensorFlow version 2.20.0 available.
08:22:11 INFO    datasets | JAX version 0.7.2 available.
08:22:12 INFO    httpx | HTTP Request: HEAD https://huggingface.co/data

0

## 7. DPO (1 outer iter, dev eval)

Iterative DPO with the smallest possible loop:
- 1 outer iteration (collect → train → eval) instead of 4
- 2 rollouts per seed instead of 4
- 1 epoch over cached pairs instead of 2
- gradient_accum=1

A LoRA adapter is saved under the run dir and synced to GCS.

In [9]:
DPO_TINY = [
    'dpo.n_outer=1',
    'dpo.n_per_seed=2',
    'dpo.n_epochs=1',
    'dpo.grad_accum=1',
]

run_method(
    method='dpo',
    run_name='demo-dpo-dev',
    overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + DPO_TINY,
)

$ python -m src dpo --config configs/dpo.yaml --run-name demo-dpo-dev --gcs-bucket gs://results_043026 --override data.n_train=4 --override data.n_dev=4 --override data.n_test=4 --override max_turns=2 --override attacker_max_new_tokens=128 --override target_max_new_tokens=128 --override judge_backend=openai --override judge_model=gpt-4o-mini --override dpo.n_outer=1 --override dpo.n_per_seed=2 --override dpo.n_epochs=1 --override dpo.grad_accum=1 --allow-dirty
08:24:07 INFO    redteam_rlhf.cli | Git SHA: 3458a65e9c7116c77a5ec2ea8fdc0b22f3655bdb
08:24:07 INFO    redteam_rlhf.cli | GPU: NVIDIA A100-SXM4-80GB
08:24:07 INFO    redteam_rlhf.cli | Seeded RNGs with seed=42
08:24:07 INFO    redteam_rlhf.cli | Run ID: ba7e7aca672c | name=demo-dpo-dev
08:24:07 INFO    redteam_rlhf.results | Run dir: results/runs/ba7e7aca672c
08:24:07 INFO    numexpr.utils | NumExpr defaulting to 12 threads.
08:24:08 INFO    datasets | TensorFlow version 2.20.0 available.
08:24:08 INFO    datasets | JAX version 0

0

## 8. PPO (2 train steps, dev eval)

PPO with the smallest possible loop:
- 2 training steps instead of 100
- 2 conversations per update instead of 4
- gradient_accumulation_steps=1
- checkpoint_every=2 (so we get one checkpoint at the end)

Final-step adapter is saved under the run dir and synced to GCS.

In [10]:
PPO_TINY = [
    'ppo.n_train_steps=2',
    'ppo.n_convos_per_update=2',
    'ppo.gradient_accumulation_steps=1',
    'ppo.checkpoint_every=2',
]

run_method(
    method='ppo',
    run_name='demo-ppo-dev',
    overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + PPO_TINY,
)

$ python -m src ppo --config configs/ppo.yaml --run-name demo-ppo-dev --gcs-bucket gs://results_043026 --override data.n_train=4 --override data.n_dev=4 --override data.n_test=4 --override max_turns=2 --override attacker_max_new_tokens=128 --override target_max_new_tokens=128 --override judge_backend=openai --override judge_model=gpt-4o-mini --override ppo.n_train_steps=2 --override ppo.n_convos_per_update=2 --override ppo.gradient_accumulation_steps=1 --override ppo.checkpoint_every=2 --allow-dirty
08:29:24 INFO    redteam_rlhf.cli | Git SHA: 3458a65e9c7116c77a5ec2ea8fdc0b22f3655bdb
08:29:24 INFO    redteam_rlhf.cli | GPU: NVIDIA A100-SXM4-80GB
08:29:24 INFO    redteam_rlhf.cli | Seeded RNGs with seed=42
08:29:24 INFO    redteam_rlhf.cli | Run ID: fbd30e9afdf6 | name=demo-ppo-dev
08:29:24 INFO    redteam_rlhf.results | Run dir: results/runs/fbd30e9afdf6
08:29:28 INFO    numexpr.utils | NumExpr defaulting to 12 threads.
08:29:29 INFO    datasets | TensorFlow version 2.20.0 available.
0

0

## 9. Test-split evaluation

Adding `--use-test` to any method flips the final eval from the dev split to the held-out test split. For `dpo` / `ppo` this re-runs training as well — that's how the system is wired (training and final eval happen in the same invocation). For paper-grade numbers you'd run each method **once** with `--use-test` directly.

Below we just demo the test-eval flag on `baseline` (no training, so it's fast).

In [11]:
run_method(
    method='baseline',
    run_name='demo-baseline-test',
    overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE,
    use_test=True,
)

$ python -m src baseline --config configs/baseline.yaml --run-name demo-baseline-test --gcs-bucket gs://results_043026 --override data.n_train=4 --override data.n_dev=4 --override data.n_test=4 --override max_turns=2 --override attacker_max_new_tokens=128 --override target_max_new_tokens=128 --override judge_backend=openai --override judge_model=gpt-4o-mini --use-test --allow-dirty
08:31:29 INFO    redteam_rlhf.cli | Git SHA: 3458a65e9c7116c77a5ec2ea8fdc0b22f3655bdb
08:31:29 INFO    redteam_rlhf.cli | GPU: NVIDIA A100-SXM4-80GB
08:31:29 INFO    redteam_rlhf.cli | Seeded RNGs with seed=42
08:31:29 INFO    redteam_rlhf.cli | Run ID: 0a033d2d8059 | name=demo-baseline-test
08:31:29 INFO    redteam_rlhf.results | Run dir: results/runs/0a033d2d8059
08:31:29 INFO    numexpr.utils | NumExpr defaulting to 12 threads.
08:31:29 INFO    datasets | TensorFlow version 2.20.0 available.
08:31:29 INFO    datasets | JAX version 0.7.2 available.
08:31:31 INFO    httpx | HTTP Request: HEAD https://huggin

0

If you want to test-evaluate the trained DPO / PPO attackers as part of this demo, uncomment below. Each will retrain with the tiny knobs and then eval on the test split.

In [ ]:
# run_method(
#     method='dpo',
#     run_name='demo-dpo-test',
#     overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + DPO_TINY,
#     use_test=True,
# )
#
# run_method(
#     method='ppo',
#     run_name='demo-ppo-test',
#     overrides=TINY_DATA + TINY_GEN + DEMO_JUDGE + PPO_TINY,
#     use_test=True,
# )

## 10. Inspect this demo's artifacts in GCS

In [13]:
!gsutil ls -r {GCS_BUCKET}/baseline/ | grep demo- | tail -20
!gsutil ls -r {GCS_BUCKET}/dpo/      | grep demo- | tail -20
!gsutil ls -r {GCS_BUCKET}/ppo/      | grep demo- | tail -20

Caught CTRL-C (signal 2) - exiting
Exception ignored in: <_io.TextIOWrapper name='<stdout>' mode='w' encoding='utf-8'>
BrokenPipeError: [Errno 32] Broken pipe
^C
